# 01. Разведочный анализ и корреляционный анализ данных о жилье в Бостоне

Этот ноутбук решает две задачи:

1. **Подготовка данных**: проверка и приведение типов, поиск и обработка пропусков и выбросов, конструирование новых признаков.
2. **Первичный корреляционный анализ**: сравнение корреляции Пирсона, Спирмена и Кендалла, интерпретация результатов и их ограничений.

Результат работы — очищенный датасет `housing_cleaned.csv`, который будет использоваться в ноутбуке `02_modeling.ipynb`.

## Описание признаков

| Признак | Описание |
|---|---|
| CRIM    | уровень преступности на душу населения по городам |
| ZN      | доля жилых земель, зонированных под участки > 25 000 кв. футов |
| INDUS   | доля неторговых коммерческих площадей по городам |
| CHAS    | фиктивная переменная реки Чарльз (1 — граничит с рекой, 0 — нет) |
| NOX     | концентрация оксидов азота (частей на 10 млн) |
| RM      | среднее количество комнат на жилое помещение |
| AGE     | доля жилых единиц в собственности, построенных до 1940 г. |
| DIS     | взвешенное расстояние до 5 центров занятости Бостона |
| RAD     | индекс доступности радиальных автомагистралей |
| TAX     | ставка налога на имущество на $10 000 стоимости |
| PTRATIO | соотношение учеников и учителей по городам |
| B       | 1000(Bk − 0.63)², Bk — доля чернокожего населения по городам |
| LSTAT   | процент населения с низким социальным статусом |
| PRICE   | медианная стоимость жилья, $1000 (целевая переменная) |

> **Важное этическое замечание.** Признак `B` был сконструирован авторами исходного датасета (Harrison & Rubinfeld, 1978) исходя из спорной гипотезы о влиянии расового состава района на цену жилья. Этот датасет по этой причине был удалён из `scikit-learn` начиная с версии 1.2. Мы оставляем признак в анализе из учебных соображений (демонстрация корреляционного анализа и работы с реальными архивными данными), но явно указываем на эту проблему и не используем её как повод для каких-либо содержательных выводов о причинно-следственных связях.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

np.random.seed(42)

## 1. Загрузка данных и проверка типов

Файл `housing.csv` не содержит строки заголовка и разделён произвольным числом пробелов (это классический формат исходного датасета Boston Housing), поэтому обычный `pd.read_csv(..., sep=",")` не подойдёт — нужно использовать регулярное выражение `\s+` в качестве разделителя и явно задать имена столбцов.


In [ ]:
COLUMNS = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS",
           "RAD", "TAX", "PTRATIO", "B", "LSTAT", "PRICE"]

df_raw = pd.read_csv("housing.csv", sep=r"\s+", header=None, names=COLUMNS)
print(f"Размер датасета: {df_raw.shape[0]} строк, {df_raw.shape[1]} столбцов")
df_raw.head()

In [ ]:
df_raw.info()

**Наблюдение по типам данных.** Все столбцы уже распознаны как числовые (`float64`/`int64`), явных проблем с типами (например, чисел, сохранённых как текст) нет. Отдельно стоит отметить:

- `CHAS` — по своей природе бинарный категориальный признак (0/1), хотя хранится как `int64` — это нормально, преобразование в `category` здесь не обязательно, так как модели линейной регрессии и деревья одинаково хорошо работают с 0/1 в числовом виде.
- `RAD` — индекс доступности, принимает ограниченный набор целых значений (по сути, порядковая/категориальная шкала с выбросом на значении 24). Оставляем численным, но учитываем это при интерпретации корреляций.

Дополнительно преобразуем `CHAS` в `int8`, а `RAD` — в `int16`, чтобы явно отразить их дискретную природу и сократить память (это скорее демонстрация хорошей практики, чем необходимость на датасете из 506 строк).


In [ ]:
df = df_raw.copy()
df["CHAS"] = df["CHAS"].astype("int8")
df["RAD"] = df["RAD"].astype("int16")
df.dtypes

## 2. Проверка пропусков

Проверим явные пропуски (`NaN`) и неявные (например, значения-заглушки вроде `-1`, `999`, повторяющиеся "круглые" значения на границах диапазона, которые могут говорить о цензурировании).


In [ ]:
missing = df.isna().sum()
print("Явные пропуски (NaN) по столбцам:")
print(missing[missing > 0] if missing.sum() > 0 else "Явных пропусков нет.")

print("\nМинимальные значения по столбцам (проверка на подозрительные заглушки, например -1):")
df.min()

**Вывод.** Явных пропусков `NaN` в данных нет, подозрительных отрицательных заглушек тоже не обнаружено (минимальное значение любого признака ≥ 0). Однако стоит проверить **неявное цензурирование целевой переменной** `PRICE`: датасет собран в 1978 году, и известно, что медианная стоимость жилья в исходных данных была "обрезана" на уровне $50 000 (т.е. все дома дороже 50k получили значение ровно 50.0).


In [ ]:
n_censored = (df["PRICE"] >= 50).sum()
print(f"Число наблюдений с PRICE == 50.0: {n_censored} из {len(df)} ({n_censored/len(df):.1%})")
df.loc[df["PRICE"] >= 49.5, ["PRICE"]].describe()

**Решение по цензурированию.** 16 наблюдений (3.2%) имеют цену ровно 50.0 — это следы цензурирования целевой переменной на этапе сбора данных 1970-х годов, а не пропуски и не выбросы в привычном смысле. Удалять эти строки нельзя (это осмысленные наблюдения "дорогое жильё"), но при интерпретации метрик регрессии в ноутбуке 2 стоит помнить, что модель структурно не сможет корректно предсказывать цены выше потолка в 50 — это ограничение данных, а не модели. Оставляем эти строки в датасете без изменений.

## 3. Поиск и обработка выбросов

Для поиска выбросов используем boxplot (диаграмму размаха) по каждому непрерывному признаку. Так как масштабы признаков сильно различаются (например, `TAX` — сотни, `NOX` — доли единицы), для визуализации предварительно стандартизируем данные (z-score), но саму обработку выбросов будем выполнять на исходных (нестандартизированных) значениях.


In [ ]:
continuous_cols = ["CRIM", "ZN", "INDUS", "NOX", "RM", "AGE", "DIS", "TAX", "PTRATIO", "B", "LSTAT"]

df_z = (df[continuous_cols] - df[continuous_cols].mean()) / df[continuous_cols].std()

fig, ax = plt.subplots(figsize=(13, 5))
df_z.boxplot(ax=ax, rot=45)
ax.set_title("Boxplot стандартизированных непрерывных признаков (поиск выбросов)")
ax.set_ylabel("z-score")
plt.tight_layout()
plt.show()

**Интерпретация.** Сильнее всего за пределы "усов" (±1.5 IQR) выходят `CRIM` (уровень преступности — сильно скошенное распределение, большинство районов с низкой преступностью и небольшое число районов с очень высокой), `ZN`, `B` (большинство значений сосредоточено у верхней границы, редкие низкие выбросы) и `LSTAT` отчасти. Это ожидаемо для социально-экономических показателей по городам — такие величины почти всегда имеют распределение с тяжёлым правым хвостом, а не нормальное.

**Стратегия обработки.** На датасете из 506 наблюдений удаление строк-выбросов — рискованная стратегия: мы можем потерять важные для модели "крайние" районы (очень дешёвые/дорогие, очень криминальные и т.д.), которые часто наиболее информативны именно для регрессии. Поэтому вместо удаления используем **винзоризацию (capping) по методу IQR**: значения, выходящие за границы `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`, обрезаются до этих границ, а не удаляются. Это уменьшает влияние экстремальных значений на модели, чувствительные к выбросам (линейная регрессия, SVR), но сохраняет все строки и общую структуру данных.

Признаки `CHAS` (бинарный) и `RAD` (дискретный индекс с осмысленным выбросом-категорией 24, которая означает "максимальная доступность") из обработки исключаем — обрезка испортила бы их содержательный смысл. `PRICE` как целевую переменную также не трогаем (см. раздел про цензурирование выше) — обрабатывать выбросы в таргете до построения модели некорректно, это исказит саму задачу прогнозирования.


In [ ]:
def iqr_cap(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    return series.clip(lower=lower, upper=upper), lower, upper

cap_cols = [c for c in continuous_cols]  # все непрерывные признаки, кроме CHAS/RAD/PRICE
before = df[cap_cols].copy()

cap_bounds = {}
for col in cap_cols:
    df[col], lo, hi = iqr_cap(df[col])
    cap_bounds[col] = (round(lo, 3), round(hi, 3))

n_changed = (before != df[cap_cols]).sum()
print("Количество изменённых (обрезанных) значений по признакам:")
print(n_changed[n_changed > 0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
((before - before.mean()) / before.std()).boxplot(ax=axes[0], rot=45)
axes[0].set_title("До винзоризации")
((df[cap_cols] - df[cap_cols].mean()) / df[cap_cols].std()).boxplot(ax=axes[1], rot=45)
axes[1].set_title("После винзоризации (IQR-капинг)")
plt.tight_layout()
plt.show()

После обрезки экстремальные "усы" укоротились, но общая форма распределений сохранена — мы не изменили медианы и основную массу данных, только смягчили влияние редких крайних значений.


## 4. Логарифмирование сильно скошенных признаков

`CRIM` и `LSTAT` имеют выраженную правостороннюю асимметрию (это видно и по boxplot выше, и по коэффициенту асимметрии). Для признаков, которые в дальнейшем пойдут в линейные модели, скошенность — проблема: линейная регрессия предполагает линейную связь и не любит сильно нелинейные, "пилообразные" по плотности признаки. Добавим их логарифмированные версии как новые признаки (не заменяя оригиналы — пусть на этапе моделирования будет выбор).


In [ ]:
skew_before = df[["CRIM", "LSTAT", "ZN", "B"]].apply(lambda s: stats.skew(s))
print("Коэффициент асимметрии (skewness) до логарифмирования:")
print(skew_before.round(2))

df["CRIM_LOG"] = np.log1p(df["CRIM"])
df["LSTAT_LOG"] = np.log1p(df["LSTAT"])

skew_after = pd.Series({
    "CRIM_LOG": stats.skew(df["CRIM_LOG"]),
    "LSTAT_LOG": stats.skew(df["LSTAT_LOG"]),
})
print("\nКоэффициент асимметрии после логарифмирования (log1p):")
print(skew_after.round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
sns.histplot(df["CRIM"], kde=True, ax=axes[0, 0], color="#4C72B0")
axes[0, 0].set_title("CRIM (исходный)")
sns.histplot(df["CRIM_LOG"], kde=True, ax=axes[0, 1], color="#55A868")
axes[0, 1].set_title("log1p(CRIM)")
sns.histplot(df["LSTAT"], kde=True, ax=axes[1, 0], color="#4C72B0")
axes[1, 0].set_title("LSTAT (исходный)")
sns.histplot(df["LSTAT_LOG"], kde=True, ax=axes[1, 1], color="#55A868")
axes[1, 1].set_title("log1p(LSTAT)")
plt.tight_layout()
plt.show()

Логарифмирование заметно снизило асимметрию `CRIM` (распределение преступности растянуто на порядки — от долей процента до почти 90, поэтому лог-шкала для него естественна) и немного улучшило форму распределения `LSTAT`. Оставляем обе версии в датасете.


## 5. Конструирование новых признаков

Добавим несколько признаков, потенциально полезных для моделирования:

- **`RM2`** — квадрат среднего числа комнат. Известно, что связь `RM ↔ PRICE` нелинейна и ускоряется при большом числе комнат (роскошное жильё дорожает быстрее пропорционального роста комнат) — квадратичный член поможет линейным моделям уловить эту нелинейность без перехода к полностью непараметрическим моделям.
- **`TAX_PER_PTRATIO`** — отношение налоговой ставки к соотношению ученик/учитель. Комбинированная прокси-переменная "налогового бремени с поправкой на качество образования в районе": высокий налог при хорошем (низком) соотношении ученик/учитель воспринимается иначе, чем высокий налог при плохом соотношении.
- **`DIS_LOG`** — логарифм расстояния до центров занятости (расстояния тоже имеют правостороннюю скошенность).
- **`RAD_TAX_INTERACTION`** — `RAD * TAX`, так как ниже мы увидим, что `RAD` и `TAX` сильно коррелируют между собой (типичный кейс мультиколлинеарности из-за структуры транспортных зон), их взаимодействие может нести дополнительную информацию о "дорогих транспортных узлах".


In [ ]:
df["RM2"] = df["RM"] ** 2
df["TAX_PER_PTRATIO"] = df["TAX"] / df["PTRATIO"]
df["DIS_LOG"] = np.log1p(df["DIS"])
df["RAD_TAX_INTERACTION"] = df["RAD"] * df["TAX"]

new_feats = ["RM2", "TAX_PER_PTRATIO", "DIS_LOG", "RAD_TAX_INTERACTION", "CRIM_LOG", "LSTAT_LOG"]
df[new_feats].describe().T

## 6. Корреляционный анализ

Сравним три меры парной зависимости:

- **Пирсон (Pearson)** — измеряет силу **линейной** связи между двумя количественными переменными. Значение в диапазоне [-1, 1]. Чувствителен к выбросам и предполагает (для проверки значимости через t-критерий), что данные хотя бы приблизительно нормально распределены. **Важная оговорка**: `corr_pearson = 0` означает отсутствие *линейной* связи, но не означает независимость переменных — между ними вполне может быть сильная нелинейная зависимость (классический пример — `Y = X²` при симметричном относительно нуля `X`, где Пирсон близок к 0, хотя связь детерминирована). Строго говоря, ноль корреляции Пирсона гарантирует независимость только для совместно нормально распределённых величин — это частный случай, а не общее правило.
- **Спирмен (Spearman)** — корреляция рангов, измеряет силу **монотонной** (не обязательно линейной) связи. Устойчивее к выбросам, чем Пирсон, так как работает с рангами, а не с абсолютными значениями. Не требует нормальности распределений. Хорошо подходит для наших скошенных признаков (`CRIM`, `LSTAT`, `B`).
- **Кендалл (Kendall's tau)** — тоже ранговая корреляция, но основана на подсчёте согласованных/несогласованных пар наблюдений. Более устойчива к выбросам, чем Спирмен, и её проще корректно интерпретировать при малых выборках и большом числе связанных рангов (ties), но вычислительно дороже. Значения Кендалла обычно меньше по модулю, чем Спирмена, для одной и той же зависимости — это ожидаемо и не говорит о более слабой связи.

Построим тепловые карты для Пирсона и Спирмена по числовым признакам (без вручную сконструированных, чтобы не загромождать картинку дублирующей информацией, а также без явно избыточных интеракций).


In [ ]:
base_cols = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS",
             "RAD", "TAX", "PTRATIO", "B", "LSTAT", "PRICE"]

corr_pearson = df[base_cols].corr(method="pearson")
corr_spearman = df[base_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_pearson, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Корреляция Пирсона")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_spearman, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Корреляция Спирмена")
plt.tight_layout()
plt.show()

In [ ]:
diff = (corr_spearman - corr_pearson).abs()
top_diff = (
    diff.where(np.triu(np.ones(diff.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
    .head(10)
)
print("Топ-10 пар признаков с наибольшим расхождением |Spearman - Pearson|:")
top_diff

**Ключевые наблюдения по корреляции с целевой переменной `PRICE`:**

- `LSTAT` (доля населения с низким социальным статусом) — самая сильная отрицательная связь с ценой по обеим метрикам, при этом по Спирмену связь заметно сильнее, чем по Пирсону — это указывает на то, что зависимость `LSTAT ↔ PRICE` **монотонная, но нелинейная** (что мы визуально проверим ниже).
- `RM` (число комнат) — самая сильная положительная связь с ценой, здесь Пирсон и Спирмен близки, то есть связь скорее линейная.
- `PTRATIO` и `TAX` отрицательно связаны с ценой — районы с более высоким налогом и худшим соотношением учеников на учителя в среднем дешевле (или это прокси для более "спальных"/удалённых районов).
- `CHAS` показывает слабую, но положительную связь с ценой — близость к реке Чарльз слегка повышает стоимость, эффект небольшой и на бинарном признаке измерение через точечно-бисериальную корреляцию (совпадает с формулой Пирсона для бинарного признака) вполне корректно.

**Мультиколлинеарность между признаками (не с таргетом):**

- `RAD` и `TAX` показывают очень высокую положительную корреляцию (≈0.9) — это яркий пример мультиколлинеарности: районы с высокой доступностью радиальных шоссе почти всегда попадают в определённые налоговые зоны с высокой ставкой. Для линейной регрессии совместное включение этих признаков может приводить к нестабильным, малоинтерпретируемым коэффициентам (это разберём подробнее в ноутбуке 2 через VIF).
- `NOX` и `INDUS` также сильно коррелируют — концентрация промышленных зон закономерно связана с загрязнением воздуха.
- `DIS` отрицательно коррелирует с `NOX`, `INDUS`, `AGE` — чем дальше от центров занятости, тем чище воздух, меньше промзон и новее жильё (пригороды застраивались позже).

**Почему стоит смотреть на обе метрики одновременно**: пары с большим расхождением между Пирсоном и Спирменом (см. таблицу выше) — это кандидаты на нелинейную зависимость, которую линейная регрессия "по умолчанию" недооценит. Именно поэтому в разделе 5 мы заранее добавили `RM2`, `LOG`-версии `CRIM`/`LSTAT`/`DIS` — они дают линейным моделям шанс уловить эти нелинейности через преобразованные признаки.


### Визуальная проверка формы связи ключевых признаков с ценой

Дополним корреляционные коэффициенты диаграммами рассеяния с трендом — числовая корреляция не показывает *форму* связи, а только её силу и направление (классический пример — квартет Энскомба, где сильно разные по форме зависимости дают одинаковый коэффициент Пирсона).


In [ ]:
top_features = ["LSTAT", "RM", "PTRATIO", "TAX", "NOX", "CRIM_LOG"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), top_features):
    sns.regplot(x=df[col], y=df["PRICE"], ax=ax, scatter_kws={"alpha": 0.4, "s": 15},
                order=2, line_kws={"color": "red"})
    ax.set_title(f"{col} vs PRICE")
plt.tight_layout()
plt.show()

Красная линия — квадратичная (полиномиальная 2-й степени) аппроксимация, которая лучше отражает фактическую форму зависимости, чем прямая линия. Видно, что:

- `LSTAT vs PRICE` — явно выпуклая, нелинейная (убывающая с замедлением) зависимость — это подтверждает вывод из сравнения Пирсона/Спирмена выше и оправдывает добавление `LSTAT_LOG`.
- `RM vs PRICE` — в целом близка к линейной с ускорением в верхнем хвосте (много комнат → непропорциональный рост цены), что и должен уловить признак `RM2`.
- `TAX vs PRICE` и `PTRATIO vs PRICE` — в целом убывающие, с сильным разбросом, что отражает их посредственную (хотя и статистически значимую) предсказательную силу по отдельности.
- Верхний "потолок" в области `PRICE ≈ 50` на всех графиках — это визуальный след цензурирования таргета, отмеченного в разделе 2.


## 7. Итоги подготовки данных

Что было сделано:

1. Данные загружены с правильным разделителем и именами столбцов, типы `CHAS`/`RAD` приведены к более компактным дискретным типам.
2. Явных пропусков не обнаружено; обнаружено и задокументировано (но не устранено — это не ошибка, а особенность сбора данных) цензурирование таргета `PRICE` на уровне 50.
3. Выбросы в непрерывных признаках (кроме `CHAS`, `RAD`, `PRICE`) обработаны методом IQR-винзоризации (обрезка, а не удаление строк) — сохраняет все 506 наблюдений.
4. Сильно скошенные признаки `CRIM` и `LSTAT` дополнены логарифмированными версиями.
5. Добавлены новые признаки: `RM2`, `TAX_PER_PTRATIO`, `DIS_LOG`, `RAD_TAX_INTERACTION`.
6. Проведён корреляционный анализ (Пирсон, Спирмен), выявлены как сильные линейные связи (`RM`), так и монотонные нелинейные (`LSTAT`), а также значимая мультиколлинеарность (`RAD` ↔ `TAX`, `NOX` ↔ `INDUS`), которую нужно будет учесть при выборе и регуляризации моделей в следующем ноутбуке.

Сохраняем итоговый датасет для этапа моделирования.


In [ ]:
print("Итоговые столбцы датасета:")
print(list(df.columns))
print(f"\nИтоговый размер: {df.shape}")

df.to_csv("housing_cleaned.csv", index=False)
print("\nФайл housing_cleaned.csv сохранён.")